In [1]:
from csv import DictReader
from numpy import zeros
import numpy as np
import copy as cp
print("running program!")

#function to load in training data and establish vocabulary
def load_data(csv_file):
    vocab = {}
    vocab_size = 0
    examples = []
    with open(csv_file, 'r') as f:
        reader = DictReader(f)
        rows=0
        for row in reader:
            rows+=1
            label = row['rating'] == '1'
            words = row['text'].split(' ')
            for word in words:
                if word not in vocab:
                    vocab[word] = vocab_size
                    vocab_size += 1
            examples.append((label, [vocab[word] for word in words]))
    #print(rows)
    print('done loading')
    return (examples,vocab_size,vocab)

examples,vocab_size,vocab=load_data('data_files/reviews_tr.csv')

#function to load in test data and add test data words to vocabulary
#this wont affect anything because the words that get added arent in the training set
#so the weight vector established by perceptron will ony have 0's at the indices of these entries
#which makes behave as if its just operating with the vocabulary of the training set
def load_test_data(csv_file,vc,vs):
    vocab = vc
    vocab_size = vs
    examples = []
    #print(vs)
    with open(csv_file, 'r') as f:
        reader = DictReader(f)
        rows=0
        for row in reader:
            rows+=1
            label = row['rating'] == '1'
            words = row['text'].split(' ')
            for word in words:
                if word not in vocab:
                    #print("made not words")
                    vocab[word] = vocab_size
                    vocab_size += 1
            examples.append((label, [vocab[word] for word in words]))
    #print(vocab_size)
    print('done test loading')
    return (examples,vocab_size,vocab)

test_examples,vocab_size,vocab = load_test_data('data_files/reviews_te.csv',vocab,vocab_size)

#conceptual bag of wors representation
def bag_of_words_rep(word_ids, dim):
    bow_vector = zeros(dim) # creates a numpy.ndarray of shape (dim,)
    for word_id in word_ids:
        bow_vector[word_id] += 1
    return bow_vector
#first_bow_vector = bag_of_words_rep(examples[0][1], vocab_size)
#inverting the vocab map for later
inv_vocab = {v: k for k, v in vocab.items()}
#print(first_bow_vector)
#print(inv_vocab[0])

#print(vocab_size,'\n',examples[10])
#print out the review in text form
def print_Review():
    verbal_rev=[]
    for i in examples[10][1]:
        verbal_rev.append(inv_vocab[i])
    print(verbal_rev)
#print_Review()

#instead of building a bag-of-words vector for every case we can just make a dictionary 
#that associates the multiplicity of a word with its index inside the bag of words vector


#this functions actually not needed
def parse_bow(loc_bow_vector,index):
    if index in loc_bow_vector:
        return loc_bow_vector[index]
    else:
        return 0
#print(parse_bow({1:3,5:1,4:2},5))

#x is a dictionary and w is a vector of all the words
def dot_p_xw(x,w):
    #dot product is just sum(x_i * w_i)
    product_sum=0
    for x_i in x: #initial i made mistake of parsing through w which meant 200 billion iterations 
        #print(i, x[i])
        product_sum+=x[x_i]*w[x_i]
    return product_sum
#print(dot_p_xw({1:3,5:1,4:2},[1,2,1,1,1,1]))
    
#logit functions to compress y output to only being between 0 and 1
def pr_y_is_1(x,w):
    f = 1/(np.exp(dot_p_xw(x,w))) + 1
    return 1/f
def pr_y_is_0(x,w):
    f = 1 - pr_y_is_1(x,w)
    return f

#shortent version of bag of word
#instead of having a R^p vector with zeros at most entries ive just turned x into a dictionary that contains
#value of how many times a word shows up with its key being the index so this way Ive just catured the useful 
#information without the length
def bag_of_words_rep_short(word_ids):
    bow_vector_short={}
    for word_id in word_ids:
        #print(word_id)
        #bow_vector[word_id] += 1
        if word_id in bow_vector_short:
            bow_vector_short[word_id]+=1
        elif word_id not in bow_vector_short:
            bow_vector_short[word_id]=1
    return bow_vector_short
weight_vector = zeros(vocab_size) #creates a numpy.ndarray of shape (dim,)

#bow = bag_of_words_rep_short(examples[0][1])
#print(first_bow_vector)
#print(pr_y_is_1(bow,weight_vector))

#converte examples from an array of tuples with 0th element = t/f and 1st element = 
#array of ideas of each word in review to --> array of tuples with 0th element = t/f
#and 1st element is the shortend bow vector
def convert_data(ex,vs,vc):
    short_bows=[]
    for x_i in ex:
        short_bows.append((x_i[0],bag_of_words_rep_short(x_i[1])))
    print('done converting')
    return short_bows
converted_data=convert_data(examples,vocab_size,vocab)
#print(converted_data[0])

#print(converted_data[10])

#optimal predictor function
def opt_pred(x,w):
    if pr_y_is_1(x,w)>=.5:
        return True
    else:
        return False

#function to update the weight vector inside the perceptron algorithm
def update_weight(tf,x,w):
    for x_i in x:
        if tf==True:
            w[x_i]+=x[x_i]
        else:
            w[x_i]-=x[x_i]
    return w
        
#print(update_weight(False,{2:8,1:3},[0,0,-1]))

#function to run perceptron to train a weight vector
def run_perceptron(conv_ex,vs):
    w_vec = zeros(vs) #creates a numpy.ndarray of shape (dim,)
    k=0
    print('running perceptron')
    for x_i in conv_ex:
        if opt_pred(x_i[1],w_vec)!=x_i[0]:
            k+=1
            if x_i[0]==True:
                w_vec=update_weight(True,x_i[1],w_vec)
            else:
                w_vec=update_weight(False,x_i[1],w_vec)
        else:
            w_vec = w_vec
    #print(k)
    print('done perceptronning')
    return w_vec

final_weight=run_perceptron(converted_data,vocab_size)
#print(final_weight)

#get test error rate
def error_rate(conv_ex,w):
    errors=0
    total=0
    for x_i in conv_ex:
        if opt_pred(x_i[1],w)!=x_i[0]:
            errors+=1
        total+=1
    return errors/total
err=error_rate(converted_data,final_weight)
print('\n\nTRAINING ERROR RATE:',err)

converted_test_data=convert_data(test_examples,vocab,vocab_size)

err=error_rate(converted_test_data,final_weight)
print('\nTEST ERROR RATE:',err)

inv_vocab = {v: k for k, v in vocab.items()}
def top10words():
    final_w_copy = cp.deepcopy(final_weight)
    #print(final_w_copy)
    #print(len(final_w_copy))
    top=[]
    bottom=[]
    for i in range(0,10):
        maxindex = int(np.where(final_w_copy==np.max(final_w_copy))[0][0])
        top.append(inv_vocab[maxindex])
        #print(inv_vocab[maxindex],maxindex,np.where(final_w_copy==np.max(final_w_copy))[0][0],np.max(final_w_copy))
        final_w_copy[maxindex]=0
    for i in range(0,10):
        minindex = int(np.where(final_w_copy==np.min(final_w_copy))[0][0])
        #print(minindex)
        bottom.append(inv_vocab[minindex])
        #print(top,minindex)
        final_w_copy[minindex]=0
    #print(top)
    return top,bottom
tsbs=top10words()
print('\n\n','TOP 10 GOOD WORDS:\n',tsbs[0],'\n\n','TOP 10 BAD WORDS\n',tsbs[1],'\n')
    
def classify_my_rev(rating,review):
    words=review.split(' ')
    ex=((rating,bag_of_words_rep_short([vocab[word] for word in words])))
    print(ex)
    #print(convert_data(examples[0][1],vocab_size,vocab))
    print("predicted rating:",opt_pred(ex[1],final_weight))
    return
#iterestingly long negative reviews get classifies as positive
#classify_my_rev(True,"good")

running program!
done loading
done test loading
done converting
running perceptron


/var/folders/gc/_l4905hj3313wyqyt96gfmj40000gn/T/ipykernel_17381/848290551.py:103: RuntimeWarning: overflow encountered in exp
  f = 1/(np.exp(dot_p_xw(x,w))) + 1
/var/folders/gc/_l4905hj3313wyqyt96gfmj40000gn/T/ipykernel_17381/848290551.py:103: RuntimeWarning: overflow encountered in scalar divide
  f = 1/(np.exp(dot_p_xw(x,w))) + 1
/var/folders/gc/_l4905hj3313wyqyt96gfmj40000gn/T/ipykernel_17381/848290551.py:103: RuntimeWarning: divide by zero encountered in scalar divide
  f = 1/(np.exp(dot_p_xw(x,w))) + 1


done perceptronning


TRAINING ERROR RATE: 0.178736
done converting

TEST ERROR RATE: 0.1793441250523238


 TOP 10 GOOD WORDS:
 ['perfection', 'disappoint', 'gem', 'phenomenal', 'perfectly', 'heavenly', 'fantastic', 'perfect', 'awesome', 'skeptical'] 

 TOP 10 BAD WORDS
 ['worst', 'mediocre', 'meh', 'underwhelmed', 'worse', 'bland', 'tasteless', 'flavorless', 'hopes', 'lacked'] 

